# Sentiment Analysis
MACS 30113 Final Project — Analyzing and Modeling part, Anyi Li

Loads preprocessed Reddit data, applies VADER (rule-based) and SparkNLP (deep learning) sentiment scoring, and saves the scored DataFrame to S3 for downstream analysis.

In [1]:
%%configure -f
{
    "conf": {
        "spark.pyspark.python": "python3",
        "spark.pyspark.virtualenv.enabled": "true",
        "spark.pyspark.virtualenv.type": "native",
        "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv"
    }
}

In [2]:
sc.install_pypi_package('vaderSentiment', 'https://pypi.org/simple')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
6,application_1780015888021_0008,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
spark

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## 1. Load Data

In [4]:
from pyspark.sql.functions import col, lit

posts_nlp = spark.read.parquet("s3://30113-final-project/reddit/posts_nlp/")
comments_nlp = spark.read.parquet("s3://30113-final-project/reddit/comments_nlp/")

# The source path is one subreddit. Keep the field explicit for downstream notebooks.
if "subreddit" not in posts_nlp.columns:
    posts_nlp = posts_nlp.withColumn("subreddit", lit("mentalhealth"))
if "subreddit" not in comments_nlp.columns:
    comments_nlp = comments_nlp.withColumn("subreddit", lit("mentalhealth"))

# Keep sentiment inputs narrow so the full-data VADER job does not move large token arrays.
sentiment_input_cols = ["subreddit", "year", "month", "created_utc", "clean_text"]
common_cols = [c for c in sentiment_input_cols if c in posts_nlp.columns and c in comments_nlp.columns]

reddit_df = posts_nlp.select(common_cols).unionByName(comments_nlp.select(common_cols))
reddit_df = reddit_df.filter(col("clean_text").isNotNull() & (col("clean_text") != ""))

print("Total records:", reddit_df.count())
reddit_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total records: 2211334
root
 |-- subreddit: string (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- clean_text: string (nullable = true)

In [5]:
reddit_df.select("subreddit", "year", "month", "clean_text").show(5, truncate=80)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+----+-----+--------------------------------------------------------------------------------+
|   subreddit|year|month|                                                                      clean_text|
+------------+----+-----+--------------------------------------------------------------------------------+
|mentalhealth|2023|    3|hey all its feelings friday  


heres a checkin writing activity for you 

 h...|
|mentalhealth|2023|    3|i just feel so lost im  and turning  in  months i cant talk to people and be ...|
|mentalhealth|2023|    3|entering week seven of a recovery from a neck sprainstrain from weightlifting...|
|mentalhealth|2023|    3|good morning friends  


what are  things you plan to do this week to practic...|
|mentalhealth|2023|    3|i think i am slightly mentally unstable for the past few weeks but i feel lik...|
+------------+----+-----+--------------------------------------------------------------------------------+
only showing top 5 rows

In [6]:
from pyspark.sql.functions import col, min, max

print("Date range:")
reddit_df.select(min("year"), max("year")).show()

print("Total records:", reddit_df.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Date range:
+---------+---------+
|min(year)|max(year)|
+---------+---------+
|     2019|     2024|
+---------+---------+

Total records: 2211334

## 2. Define COVID Event Windows
Pre-COVID: before March 2020. Post-COVID: March 2020 onward (WHO pandemic declaration: March 11, 2020).

In [7]:
from pyspark.sql.functions import when, lit

reddit_df = reddit_df.withColumn(
    "period",
    when(
        (col("year") < 2020) | ((col("year") == 2020) & (col("month") < 3)),
        lit("pre_covid")
    ).otherwise(lit("post_covid"))
)

reddit_df.groupBy("period").count().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-------+
|    period|  count|
+----------+-------+
| pre_covid| 252321|
|post_covid|1959013|
+----------+-------+

## 3. VADER Sentiment Scoring
VADER returns a compound score in [-1, +1]: negative toward -1, positive toward +1, neutral near 0.

In [8]:
reddit_df = reddit_df.sample(False, 0.10, seed=42)
print("Sample applied")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sample applied

In [9]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_vader_analyzer = None

def get_vader_sentiment(text):
    global _vader_analyzer
    if text is None:
        return 0.0
    text = text.strip()
    if text == "":
        return 0.0
    if _vader_analyzer is None:
        _vader_analyzer = SentimentIntensityAnalyzer()
    return float(_vader_analyzer.polarity_scores(text)["compound"])

vader_udf = udf(get_vader_sentiment, FloatType())

reddit_df = reddit_df.withColumn("vader_sentiment", vader_udf(col("clean_text")))

# Repartition before the full write so the UDF work is spread across the cluster.
default_partitions = sc.defaultParallelism * 4
num_output_partitions = 64 if default_partitions < 64 else default_partitions
reddit_df = reddit_df.repartition(num_output_partitions, "year", "month")

print("VADER scoring complete.")
reddit_df.select("subreddit", "period", "clean_text", "vader_sentiment").show(5, truncate=60)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

VADER scoring complete.
+------------+----------+------------------------------------------------------------+---------------+
|   subreddit|    period|                                                  clean_text|vader_sentiment|
+------------+----------+------------------------------------------------------------+---------------+
|mentalhealth|post_covid|
if you or someone you know is contemplating suicide plea...|         -0.802|
|mentalhealth|post_covid|yuuuuuup  bottling it all up and pretending im okay becau...|        -0.5574|
|mentalhealth|post_covid|i definitely believe i have an anxiety disorder it gets s...|         0.2618|
|mentalhealth|post_covid|thank you for for sharing a reminder if you are seeking r...|          0.955|
|mentalhealth|post_covid|maybe instead of desperately trying to keep conversation ...|        -0.3818|
+------------+----------+------------------------------------------------------------+---------------+
only showing top 5 rows

In [10]:
# Checkpoint after VADER scoring.
reddit_df.write.mode("overwrite").partitionBy("period").parquet(
    "s3://30113-final-project/results/sentiment_scored_checkpoint/"
)
print("Checkpoint saved.")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Checkpoint saved.

In [11]:
# Reload from S3
reddit_df = spark.read.parquet("s3://30113-final-project/results/sentiment_scored_checkpoint/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
# Categorize into positive / negative / neutral
# Threshold of 0.05 follows standard VADER convention
reddit_df = reddit_df.withColumn(
    "vader_label",
    when(col("vader_sentiment") >= 0.05, lit("positive"))
    .when(col("vader_sentiment") <= -0.05, lit("negative"))
    .otherwise(lit("neutral"))
)

reddit_df.groupBy("period", "vader_label").count().orderBy("period", "vader_label").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+-----------+------+
|    period|vader_label| count|
+----------+-----------+------+
|post_covid|   negative| 61712|
|post_covid|    neutral| 16610|
|post_covid|   positive|117838|
| pre_covid|   negative|  9189|
| pre_covid|    neutral|  2578|
| pre_covid|   positive| 13968|
+----------+-----------+------+

## 4. Incomplete SparkNLP Sentiment Labeling
SparkNLP's SentimentDL is a deep learning classifier pretrained on social media text. It uses Universal Sentence Encoder (USE) embeddings as input and outputs a positive/negative label per document. Unlike VADER, it captures semantic context rather than relying on a lexicon.

I originally planned to use SparkNLP SentimentDL as a second sentiment model, but this was not feasible in the EMR environment because SparkNLP and transformer-based models required additional system dependencies and configuration that were not available. To keep the later notebooks consistent with the original workflow, I keep a `sparknlp_sentiment` column, but it stores the same label as the VADER-based sentiment label.

In [13]:
# Note: SparkNLP and HuggingFace transformers require additional system
# dependencies (Rust compiler, JAR configuration) not available in EMR-6.2.0
# VADER sentiment analysis is used as the primary method, which is well-validated
# for social media text (Hutto & Gilbert, 2014)

# Create sparknlp_sentiment as a copy of vader_label for pipeline consistency
from pyspark.sql.functions import col
reddit_df = reddit_df.withColumn("sparknlp_sentiment", col("vader_label"))

print("Sentiment analysis complete using VADER.")
reddit_df.select("clean_text", "vader_sentiment", "sparknlp_sentiment").show(5, truncate=60)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Sentiment analysis complete using VADER.
+------------------------------------------------------------+---------------+------------------+
|                                                  clean_text|vader_sentiment|sparknlp_sentiment|
+------------------------------------------------------------+---------------+------------------+
|ive had several surgeries in my  years of life at least  ...|        -0.9809|          negative|
|lately i feel that i dont care about the world or i prete...|        -0.6875|          negative|
|hello friends basically on days where i dont take my medi...|        -0.8971|          negative|
|i dont know how to pull myself out with work stress and h...|        -0.9926|          negative|
|hey everyone i m been living by myself in the past few mo...|         0.6001|          positive|
+------------------------------------------------------------+---------------+------------------+
only showing top 5 rows

## 5. Sentiment Label Summary

This section summarizes the VADER-based sentiment labels by period. Since SparkNLP could not be run in the EMR environment, the `sparknlp_sentiment` column is kept only for compatibility with the rest of the pipeline and is equal to the VADER label.

In [14]:
total = reddit_df.count()
agree = reddit_df.filter(col("vader_label") == col("sparknlp_sentiment")).count()
print(f"Agreement rate: {agree/total*100:.1f}%")

print("\nSparkNLP sentiment by period:")
reddit_df.groupBy("period", "sparknlp_sentiment").count().orderBy("period", "sparknlp_sentiment").show()

print("VADER label by period:")
reddit_df.groupBy("period", "vader_label").count().orderBy("period", "vader_label").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Agreement rate: 100.0%

SparkNLP sentiment by period:
+----------+------------------+------+
|    period|sparknlp_sentiment| count|
+----------+------------------+------+
|post_covid|          negative| 61712|
|post_covid|           neutral| 16610|
|post_covid|          positive|117838|
| pre_covid|          negative|  9189|
| pre_covid|           neutral|  2578|
| pre_covid|          positive| 13968|
+----------+------------------+------+

VADER label by period:
+----------+-----------+------+
|    period|vader_label| count|
+----------+-----------+------+
|post_covid|   negative| 61712|
|post_covid|    neutral| 16610|
|post_covid|   positive|117838|
| pre_covid|   negative|  9189|
| pre_covid|    neutral|  2578|
| pre_covid|   positive| 13968|
+----------+-----------+------+

## 6. Save to S3
Saves the full sentiment-scored DataFrame. Notebook 3 (event study) reads from this output.

In [15]:
reddit_df.write.mode("overwrite").partitionBy("period").parquet(
    "s3://30113-final-project/results/sentiment_scored/"
)
print("Saved to s3://30113-final-project/results/sentiment_scored/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Saved to s3://30113-final-project/results/sentiment_scored/